# NA-MPNN Specificity

![NA-MPNN Specificity](https://proto-bio.github.io/proto-assets/images/tool/na_mpnn_specificity/hero.png)

[NA-MPNN](https://github.com/baker-laboratory/NA-MPNN) predicts protein–DNA specificity from a protein–DNA complex structure. The `na-mpnn-specificity` tool returns a canonical DNA-only position probability matrix (PPM) in `A,C,G,T` order for each input structure, along with the recovered true sequence, per-row masks, chain labels, and, when `output_directory` is set, a path to a canonical `.npz` artifact.

Preprint: [Kubaney et al., 2025](https://www.biorxiv.org/content/10.1101/2025.10.03.679414v2)

> The NA-MPNN checkout and its in-tree specificity checkpoint are provisioned automatically into the managed weights cache by the standalone setup; no repository or checkpoint path is configured. Run on a GPU host.

In [1]:
from pathlib import Path

from proto_tools.tools.sequence_scoring.na_mpnn_specificity import (
    NAMPNNSpecificityConfig,
    NAMPNNSpecificityInput,
    run_na_mpnn_specificity,
)
from proto_tools.utils.notebook_docs import display_api_reference

## Input API Reference

In [2]:
display_api_reference("na-mpnn-specificity", "input", "run_na_mpnn_specificity")

**Input** — `NAMPNNSpecificityInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>pdb_paths</code> | <code>list[str]</code> | required | PDB file paths for protein-DNA complexes to score for DNA specificity |

## Config API Reference

In [3]:
display_api_reference("na-mpnn-specificity", "config", "run_na_mpnn_specificity")

**Config** — `NAMPNNSpecificityConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>device</code> | <code>str</code> | <code>'cuda'</code> | Device to run NA-MPNN inference on |
| <code>batch_size</code> | <code>int</code> | <code>8</code> | Sequences per GPU forward pass; raise for throughput, lower if OOM |
| <code>number_of_batches</code> | <code>int</code> | <code>1</code> | Number of NA-MPNN prediction batches to run |
| <code>temperature</code> | <code>float</code> | <code>0.1</code> | Sampling temperature for NA-MPNN |
| <code>omit_aa</code> | <code>str</code> | <code>'X'</code> | Residue tokens to omit, concatenated with no separator (e.g. 'XCG'); case-sensitive |
| <code>design_na_only</code> | <code>bool</code> | <code>True</code> | Restrict design to nucleic acid positions |
| <code>output_directory</code> | <code>str &#124; None</code> | <code>None</code> | Directory to keep canonical NPZ files in; when unset, nothing is kept |
| <code>keep_intermediate</code> | <code>bool</code> | <code>False</code> | Keep raw NA-MPNN output under output_directory (requires it) |

## Output API Reference

In [4]:
display_api_reference("na-mpnn-specificity", "output", "run_na_mpnn_specificity")

**Output** — `NAMPNNSpecificityOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>results</code> | <code>list[NAMPNNSpecificityResult]</code> | required | Canonicalized specificity result per input structure |

**`NAMPNNSpecificityResult`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>input_name</code> | <code>str</code> | required | Basename of the scored input structure |
| <code>source_method</code> | <code>str</code> | required | Source method tag for this prediction |
| <code>output_npz_path</code> | <code>str &#124; None</code> | required | Path to the canonical NPZ file; None unless output_directory is set |
| <code>predicted_ppm</code> | <code>list[list[float]]</code> | required | Canonical DNA PPM (L x 4) in A,C,G,T order |
| <code>true_sequence</code> | <code>list[int]</code> | required | Canonical DNA truth indices in 0..3 |
| <code>mask</code> | <code>list[int]</code> | required | Valid residue mask for canonical rows |
| <code>dna_mask</code> | <code>list[int]</code> | required | DNA residue mask for canonical rows |
| <code>chain_labels</code> | <code>list[int]</code> | required | Canonical chain IDs for canonical rows |

## Basic Usage

Score a protein–DNA complex for DNA specificity. Replace the path with a real protein–DNA complex PDB on your machine.

In [5]:
# The protein-DNA complex bundled with the toolkit.
pdb_path = str(Path("..") / "example_input_fixture.pdb")

inputs = NAMPNNSpecificityInput(pdb_paths=[pdb_path])
config = NAMPNNSpecificityConfig()

result = run_na_mpnn_specificity(inputs, config)

first = result.results[0]
print("input:", first.input_name)
print("PPM shape:", len(first.predicted_ppm), "x", len(first.predicted_ppm[0]))
print("first PPM row (A,C,G,T):", first.predicted_ppm[0])
print("chain labels:", first.chain_labels)
print("npz path:", first.output_npz_path)

Running run_na_mpnn_specificity [00:00]

input: example_input_fixture
PPM shape: 20 x 4
first PPM row (A,C,G,T): [0.6104826952992033, 2.735933069101731e-08, 0.38951719085060854, 8.649085740987196e-08]
chain labels: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
npz path: None


## Advanced Usage

Tune the sampling temperature and batch count, and persist canonical `.npz` artifacts to a chosen directory.

In [6]:
config = NAMPNNSpecificityConfig(
    temperature=0.2,
    number_of_batches=2,
    output_directory="./na_mpnn_results",
    keep_intermediate=True,
)

result = run_na_mpnn_specificity(
    NAMPNNSpecificityInput(pdb_paths=[pdb_path]),
    config,
)
print("num results:", len(result.results))

Running run_na_mpnn_specificity [00:00]

num results: 1


## Export

Export the canonical results to JSON or CSV.

In [7]:
result.export("na_mpnn_specificity_results", file_format="json")
print("Exported to na_mpnn_specificity_results.json")

Exported to na_mpnn_specificity_results.json
